# GraphBench v2 — Babelscape/rebel-dataset

## ⚡ START HERE — Read before running anything

### Run order (fresh Colab session):
1. **Run Cell 1a** (install deps) → runtime will **auto-restart** — this is expected, not a crash
2. After restart, **start from Cell 1b** and run cells top-to-bottom
3. **Do NOT re-run Cell 1a** after restart (packages are already installed)

### Why the restart in Cell 1a?
Colab ships with numpy 2.x, but `sentence-transformers`/`datasets` downgrades it to
numpy 1.x, which breaks pre-compiled C extensions (ABI mismatch → `ValueError: numpy.dtype
size changed`). Cell 1a pins `numpy>=2.0` last then force-restarts so numpy 2.x is the
first thing loaded in the new session.

---

## Changes vs graphbench_main.ipynb (all bugs fixed)

| Bug | Fix |
|-----|-----|
| Install from PyPI (missing Cypher fix) | `pip install git+https://github.com/...` |
| `embed_entities` return unpacking | Returns `ndarray` only; `entity_names` built separately |
| FAISS save/load mismatch | `build_and_save_index()` → `.faiss` + `_id_map.json` |
| `write_triples` arg order | `write_triples(triples, neo4j, ...)` |
| `train_gnn` param name | `epochs=200` (not `max_epochs=`) |
| LLM 4-bit quant fails on H100 | fp16 via `_load_fp16` patch, no bitsandbytes |
| Patch returns None | `return pipe` (not `self._hf_pipeline = pipe`) |
| Deprecated kwarg | `dtype=torch.float16` (not `torch_dtype=`) |
| State lost on restart | `faiss_client` + `embedding_dict` rebuilt in Phase 4 |
| Read-only HF token | `userdata.get('graphcomp')` write token |
| Wrong HF Space ID | `rohanjain2312/graphbench-demo` |
| numpy ABI mismatch | Pin `numpy>=2.0` + runtime restart in Cell 1a |

## Phase 1 — Setup

In [ ]:
# Cell 1a — Install dependencies
# GitHub install picks up the Cypher f-string fix (Neo4j rejects $k in hop count)
#
# IMPORTANT: This cell restarts the runtime automatically at the end.
# After restart, start from Cell 1b — do NOT re-run Cell 1a.
#
# Why the restart? sentence-transformers / datasets can downgrade numpy from 2.x
# to 1.x, which breaks Colab's pre-compiled C extensions (ABI mismatch).
# We pin numpy>=2.0 last, then restart so a fresh ABI-compatible numpy is loaded.
!pip install git+https://github.com/Rohanjain2312/graphbench.git -q
!pip install datasets sentence-transformers faiss-gpu wandb -q
!pip install 'numpy>=2.0,<3.0' -q  # pin AFTER other installs so nothing downgrades it

# Restart runtime so numpy 2.x is the first numpy loaded into memory
import os
os.kill(os.getpid(), 9)

In [ ]:
# Cell 1a-check — Run this FIRST after the restart (skip if numpy>=2.0 shows below)
# Confirms packages installed correctly and numpy ABI is clean.
import numpy as np
import torch
print(f'numpy  : {np.__version__}  (need >=2.0)')
print(f'torch  : {torch.__version__}')
assert tuple(int(x) for x in np.__version__.split('.')[:1]) >= (2,), \
    f'numpy {np.__version__} is too old — re-run Cell 1a or manually: !pip install "numpy>=2.0" -q'
print('✅ numpy version OK — proceed with remaining cells')

In [ ]:
# Cell 1b — Mount Drive and define all paths upfront
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_BASE    = Path('/content/drive/MyDrive/graphbench')
CHECKPOINT_DIR = DRIVE_BASE / 'checkpoints'
RESULTS_DIR   = DRIVE_BASE / 'results'
FAISS_BASE    = DRIVE_BASE / 'faiss' / 'entities'  # → entities.faiss + entities_id_map.json
TRIPLES_PATH  = DRIVE_BASE / 'triples_babelscape.parquet'
ENV_PATH      = DRIVE_BASE / '.env'

for d in [CHECKPOINT_DIR, RESULTS_DIR, FAISS_BASE.parent]:
    d.mkdir(parents=True, exist_ok=True)

print('Drive paths ready:')
for name, p in [('CHECKPOINT_DIR', CHECKPOINT_DIR), ('RESULTS_DIR', RESULTS_DIR),
                ('FAISS_BASE', FAISS_BASE), ('TRIPLES_PATH', TRIPLES_PATH)]:
    print(f'  {name}: {p}')

In [ ]:
# Cell 1c — Load .env from Drive
import os
from dotenv import load_dotenv

load_dotenv(ENV_PATH)
print('NEO4J_URI set:', bool(os.environ.get('NEO4J_URI')))
print('HF_TOKEN set: ', bool(os.environ.get('HF_TOKEN')))
print('WANDB_API_KEY set:', bool(os.environ.get('WANDB_API_KEY')))

In [ ]:
# Cell 1d — Set FAISS and checkpoint paths in environment so settings picks them up
os.environ['FAISS_INDEX_PATH'] = str(FAISS_BASE)
os.environ['CHECKPOINT_DIR']   = str(CHECKPOINT_DIR)
print('FAISS_INDEX_PATH:', os.environ['FAISS_INDEX_PATH'])
print('CHECKPOINT_DIR:  ', os.environ['CHECKPOINT_DIR'])

In [ ]:
# Cell 1e — Verify GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU:    {torch.cuda.get_device_name(0)}')
    print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 1f — Init Neo4j and verify connectivity
from graphbench.utils.neo4j_client import Neo4jClient

neo4j = Neo4jClient()
triple_count = neo4j.count_triples()
print(f'Neo4j connected. Current triple count: {triple_count:,}')

# Cell 2a — Load and filter triples from Babelscape/rebel-dataset
# The `triplets` field is a linearized STRING, not a list of dicts:
#   "<triplet> head <subj> tail <obj> relation <triplet> head2 ..."
# We parse it before filtering.
from datasets import load_dataset
from graphbench.ingestion.rebel_loader import TOP_50_RELATIONS


def _parse_rebel_string(s: str) -> list[dict]:
    """Parse REBEL linearized triplets string into list of {head, tail, type} dicts."""
    results = []
    for chunk in s.split('<triplet>'):
        chunk = chunk.strip()
        if not chunk or '<subj>' not in chunk or '<obj>' not in chunk:
            continue
        try:
            head_part, rest = chunk.split('<subj>', 1)
            tail_part, rel_part = rest.split('<obj>', 1)
            results.append({
                'head': head_part.strip(),
                'tail': tail_part.strip(),
                'type': rel_part.strip(),
            })
        except ValueError:
            continue
    return results


MAX_TRIPLES = 60_000

if TRIPLES_PATH.exists():
    import pandas as pd
    triples = pd.read_parquet(TRIPLES_PATH).to_dict('records')
    print(f'Loaded {len(triples):,} triples from checkpoint: {TRIPLES_PATH}')
else:
    print('Downloading Babelscape/rebel-dataset (streaming) — may take ~20 min on CPU ...')
    ds = load_dataset('Babelscape/rebel-dataset', split='train', streaming=True)

    triples: list[dict] = []
    seen: set[tuple] = set()

    for ex in ds:
        raw = ex.get('triplets', '')
        # Field is a linearized string — parse it into dicts
        parsed = _parse_rebel_string(raw) if isinstance(raw, str) else (raw or [])

        for t in parsed:
            subj = t['head'].strip().lower()
            rel  = '_'.join(t['type'].strip().lower().split())  # normalise to snake_case
            obj  = t['tail'].strip().lower()

            if rel not in TOP_50_RELATIONS or not subj or not obj:
                continue
            key = (subj, rel, obj)
            if key not in seen:
                seen.add(key)
                triples.append({'subject': subj, 'relation': rel, 'object': obj})
        if len(triples) >= MAX_TRIPLES:
            break

    print(f'Loaded {len(triples):,} unique triples')

In [ ]:
# Cell 2a — Load and filter triples from Babelscape/rebel-dataset
from datasets import load_dataset
from graphbench.ingestion.rebel_loader import TOP_50_RELATIONS

MAX_TRIPLES = 60_000

if TRIPLES_PATH.exists():
    import pandas as pd
    triples = pd.read_parquet(TRIPLES_PATH).to_dict('records')
    print(f'Loaded {len(triples):,} triples from checkpoint: {TRIPLES_PATH}')
else:
    print('Downloading Babelscape/rebel-dataset (streaming) — may take ~20 min on CPU ...')
    ds = load_dataset('Babelscape/rebel-dataset', split='train', streaming=True)

    triples: list[dict] = []
    seen: set[tuple] = set()

    for ex in ds:
        for t in ex.get('triplets', []):
            subj = t['head'].strip().lower()
            # Normalise relation to snake_case (Babelscape uses spaces)
            rel  = '_'.join(t['type'].strip().lower().split())
            obj  = t['tail'].strip().lower()

            if rel not in TOP_50_RELATIONS or not subj or not obj:
                continue
            key = (subj, rel, obj)
            if key not in seen:
                seen.add(key)
                triples.append({'subject': subj, 'relation': rel, 'object': obj})
            if len(triples) >= MAX_TRIPLES:
                break
        if len(triples) >= MAX_TRIPLES:
            break

    print(f'Loaded {len(triples):,} unique triples')

In [ ]:
# Cell 2b — Relation distribution
import pandas as pd
from collections import Counter

rel_counts = Counter(t['relation'] for t in triples)
print('Top-15 relations:')
for rel, count in rel_counts.most_common(15):
    print(f'  {rel:<45} {count:>6,}')
print(f'\nTotal unique relations present: {len(rel_counts)}')

In [ ]:
# Cell 2c — Save parquet to Drive (checkpoint)
if not TRIPLES_PATH.exists():
    pd.DataFrame(triples).to_parquet(TRIPLES_PATH, index=False)
    print(f'Saved {len(triples):,} triples → {TRIPLES_PATH}')
else:
    print(f'Checkpoint already exists: {TRIPLES_PATH}')

In [ ]:
# Cell 2d — Clear Neo4j and write triples
# NOTE: correct arg order is write_triples(triples, neo4j_client, ...)
from graphbench.ingestion.neo4j_writer import write_triples

print('Clearing existing graph ...')
with neo4j._driver.session() as s:
    s.run('MATCH (n) DETACH DELETE n')
neo4j.ensure_schema()
print('Graph cleared and schema re-created.')

print(f'Writing {len(triples):,} triples to Neo4j ...')
write_triples(triples, neo4j, batch_size=500)

final_count = neo4j.count_triples()
print(f'Neo4j triple count: {final_count:,}')
assert final_count >= 50_000, f'Expected ≥50k triples, got {final_count:,}'

In [ ]:
# Cell 2e — Embed entities
# embed_entities() returns ndarray ONLY — entity names built separately
from graphbench.ingestion.embedder import embed_entities

entity_names = sorted({t['subject'] for t in triples} | {t['object'] for t in triples})
print(f'Unique entities: {len(entity_names):,}')

embeddings = embed_entities(entity_names, show_progress=True)  # returns ndarray
print(f'Embeddings shape: {embeddings.shape}, dtype: {embeddings.dtype}')

In [ ]:
# Cell 2f — Build FAISS index with build_and_save_index
# This saves .faiss + _id_map.json so FAISSClient.load() round-trip works
from graphbench.ingestion.faiss_writer import build_and_save_index

index = build_and_save_index(entity_names, embeddings, index_path=FAISS_BASE)
print(f'FAISS: {index.ntotal:,} vectors → {FAISS_BASE}.faiss')

In [ ]:
# Cell 2g — Load FAISSClient (verify round-trip)
from graphbench.utils.faiss_client import FAISSClient

faiss_client = FAISSClient.load(FAISS_BASE)
print(f'FAISSClient ready: {faiss_client.size:,} vectors')
assert faiss_client.size >= 30_000, f'Expected ≥30k entities, got {faiss_client.size:,}'

# Build embedding_dict for GNN and GNN-RAG pipeline
import numpy as np
embedding_dict: dict[str, np.ndarray] = dict(zip(entity_names, embeddings))
print(f'embedding_dict: {len(embedding_dict):,} entries')

## Phase 3 — GNN Training

3-layer GAT, link-prediction task on the knowledge graph.  
Must achieve test AUC-ROC > 0.75 before proceeding to Phase 4.

**Checkpoint**: if best checkpoint exists on Drive, load it directly.

In [ ]:
# Cell 3a — Build PyG dataset and splits
from graphbench.gnn.dataset import KGDataset
from graphbench.gnn.model import GATModel

ds_gnn = KGDataset(triples, embedding_dict)
train_data, val_data, test_data = ds_gnn.split()

print(f'Train edges: {train_data.edge_label_index.shape[1]:,}')
print(f'Val   edges: {val_data.edge_label_index.shape[1]:,}')
print(f'Test  edges: {test_data.edge_label_index.shape[1]:,}')

In [ ]:
# Cell 3b — Train GNN (or skip if checkpoint already exists)
from graphbench.gnn.trainer import train_gnn
from graphbench.utils.checkpoint import load_checkpoint
import glob

existing_ckpts = sorted(glob.glob(str(CHECKPOINT_DIR / 'gat_*.pt')))

if existing_ckpts:
    best_ckpt = existing_ckpts[-1]
    print(f'Found checkpoint: {best_ckpt}')
    model = GATModel()
    model = load_checkpoint(model, best_ckpt, device=device)
    print('Checkpoint loaded. Evaluating on test set ...')
    from graphbench.gnn.trainer import _evaluate
    test_loss, test_auc = _evaluate(model, test_data, device)
    results = {'test_auc': test_auc, 'test_loss': test_loss, 'best_val_auc': '(from ckpt)', 'best_epoch': '(from ckpt)'}
else:
    model = GATModel()
    results = train_gnn(
        model=model,
        train_data=train_data,
        val_data=val_data,
        test_data=test_data,
        checkpoint_dir=CHECKPOINT_DIR,
        device=device,
        epochs=200,                    # NOT max_epochs
        lr=1e-3,
        early_stopping_patience=15,
    )

print('GNN results:', results)
assert results['test_auc'] > 0.75, f"AUC {results['test_auc']:.4f} < 0.75 threshold"

In [ ]:
# Cell 3c — Load best checkpoint (always load so model is in eval mode with best weights)
existing_ckpts = sorted(glob.glob(str(CHECKPOINT_DIR / 'gat_*.pt')))
if existing_ckpts:
    best_ckpt = existing_ckpts[-1]
    model = GATModel()
    model = load_checkpoint(model, best_ckpt, device=device)
    model.eval()
    print(f'Best checkpoint loaded from: {best_ckpt}')
else:
    model.eval()
    print('Using model from training (no checkpoint found on Drive)')

## Phase 4 — Pipeline Initialisation

LLM loaded fp16 via monkey-patch — bypasses bitsandbytes 4-bit issues on H100.

**Key fixes:**
- `_load_fp16` *returns* `pipe` (original bug: set `self._hf_pipeline` inside, returned `None`)
- `dtype=torch.float16` (not `torch_dtype=`)
- `max_new_tokens=self.max_new_tokens` (not `self._max_new_tokens`)
- `faiss_client` and `embedding_dict` rebuilt here in case of runtime restart

In [ ]:
# Cell 4a — Rebuild faiss_client and embedding_dict (survive runtime restarts)
from graphbench.utils.faiss_client import FAISSClient
import numpy as np

faiss_client = FAISSClient.load(FAISS_BASE)
print(f'FAISSClient: {faiss_client.size:,} vectors')

# embedding_dict: reload from parquet + re-embed if not already in memory
try:
    _ = embedding_dict
    print(f'embedding_dict already in memory: {len(embedding_dict):,} entries')
except NameError:
    import pandas as pd
    from graphbench.ingestion.embedder import embed_entities
    triples = pd.read_parquet(TRIPLES_PATH).to_dict('records')
    entity_names = sorted({t['subject'] for t in triples} | {t['object'] for t in triples})
    embeddings = embed_entities(entity_names, show_progress=True)
    embedding_dict = dict(zip(entity_names, embeddings))
    print(f'embedding_dict rebuilt: {len(embedding_dict):,} entries')

In [ ]:
# Cell 4b — Monkey-patch LLMClient._load_hf_pipeline for fp16 (no bitsandbytes)
import types
import torch
from graphbench.utils import llm_client as _lc

def _load_fp16(self):
    """Load Mistral in fp16 — bypasses bitsandbytes 4-bit issues on H100."""
    from transformers import pipeline as hf_pipeline

    if not torch.cuda.is_available():
        raise RuntimeError('HuggingFace backend requires a CUDA GPU.')

    print(f'Loading {self._model} in fp16 ...')
    pipe = hf_pipeline(
        'text-generation',
        model=self._model,
        device_map='auto',
        dtype=torch.float16,              # NOT torch_dtype= (that arg was deprecated)
        max_new_tokens=self.max_new_tokens,  # NOT self._max_new_tokens
    )
    return pipe  # RETURN — not self._hf_pipeline = pipe (that returns None)

_lc.LLMClient._load_hf_pipeline = _load_fp16
print('LLMClient._load_hf_pipeline patched → fp16')

In [ ]:
# Cell 4c — Init shared LLM
from graphbench.utils.llm_client import LLMClient

llm = LLMClient(backend='hf')  # uses settings.llm_model = mistralai/Mistral-7B-Instruct-v0.2
print(f'LLM backend: {llm.backend}, model: {llm.model}')

In [ ]:
# Cell 4d — Init Neo4j (rebuild if restarted)
from graphbench.utils.neo4j_client import Neo4jClient
neo4j = Neo4jClient()
print(f'Neo4j: {neo4j.count_triples():,} triples')

In [ ]:
# Cell 4e — Build pipelines
from graphbench.pipelines.graphrag_pipeline import GraphRAGPipeline
from graphbench.pipelines.gnnrag_pipeline import GNNRAGPipeline

graphrag = GraphRAGPipeline(
    neo4j_client=neo4j,
    faiss_client=faiss_client,
    llm_client=llm,
)

gnnrag = GNNRAGPipeline(
    neo4j_client=neo4j,
    faiss_client=faiss_client,
    llm_client=llm,
    gat_model=model,
    entity_embeddings=embedding_dict,   # required for GNN-RAG node features
)

print(f'GraphRAG pipeline: {graphrag.name}')
print(f'GNN-RAG pipeline:  {gnnrag.name}')

In [ ]:
# Cell 4f — Smoke test: answer 5 questions with each pipeline
SMOKE_QUESTIONS = [
    'Where was Marie Curie born?',
    'Who directed The Godfather?',
    'What is the capital of France?',
    'Which company developed Python?',
    'Who wrote Hamlet?',
]

print('=== GraphRAG smoke test ===')
for q in SMOKE_QUESTIONS:
    result = graphrag.answer(q)
    print(f'  Q: {q}')
    print(f'  A: {result.predicted_answer}')

print('\n=== GNN-RAG smoke test ===')
for q in SMOKE_QUESTIONS:
    result = gnnrag.answer(q)
    print(f'  Q: {q}')
    print(f'  A: {result.predicted_answer}')

## Phase 5 — Benchmark

500 HotpotQA distractor questions (250 bridge, 250 comparison).  
Target: EM > 3.8% (original baseline on 11.5k triples). Expect 20–35% with 60k triples.

In [ ]:
# Cell 5a — Run benchmark
from graphbench.benchmark.evaluator import Evaluator

evaluator = Evaluator(
    pipeline_a=graphrag,
    pipeline_b=gnnrag,
    n_questions=500,
    seed=42,
    results_dir=RESULTS_DIR,
)

summary = evaluator.run()
print('\nBenchmark complete.')

## Phase 6 — Results & Analysis

In [ ]:
# Cell 6a — Print summary
import json
print(json.dumps(summary, indent=2))

In [ ]:
# Cell 6b — Summary table
import pandas as pd

rows = []
for pipeline_name, metrics in summary.items():
    rows.append({
        'Pipeline':       pipeline_name,
        'EM':             f"{metrics['em']:.1%}",
        'F1':             f"{metrics['f1']:.1%}",
        'Latency p50 ms': f"{metrics['latency_p50']:.0f}",
        'Latency p95 ms': f"{metrics['latency_p95']:.0f}",
        'N Questions':    metrics['n_questions'],
    })

df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))

In [ ]:
# Cell 6c — Load per-question CSV and analyse by question type
import glob

csv_files = sorted(glob.glob(str(RESULTS_DIR / '*_results.csv')))
if csv_files:
    df_results = pd.read_csv(csv_files[-1])
    print(f'Loaded results: {csv_files[-1]}')
    print(f'Shape: {df_results.shape}')
    display(df_results.head(5))
else:
    print('No CSV results found — check RESULTS_DIR path.')

In [ ]:
# Cell 6d — EM by question type
if 'df_results' in dir():
    em_cols = [c for c in df_results.columns if c.endswith('_em')]
    for col in em_cols:
        pipeline = col.replace('_em', '')
        by_type = df_results.groupby('type')[col].mean()
        print(f'\n{pipeline} EM by question type:')
        for qtype, em in by_type.items():
            print(f'  {qtype:<12}: {em:.1%}')

In [ ]:
# Cell 6e — Latency distribution plot
import matplotlib.pyplot as plt

if 'df_results' in dir():
    lat_cols = [c for c in df_results.columns if c.endswith('_latency_ms')]
    fig, ax = plt.subplots(figsize=(8, 4))
    for col in lat_cols:
        label = col.replace('_latency_ms', '')
        ax.hist(df_results[col], bins=50, alpha=0.6, label=label)
    ax.set_xlabel('Latency (ms)')
    ax.set_ylabel('Count')
    ax.set_title('Latency Distribution per Pipeline')
    ax.legend()
    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'latency_distribution.png'), dpi=150)
    plt.show()
    print('Plot saved to Drive.')

In [ ]:
# Cell 6f — EM assertion (baseline sanity check)
for pipeline_name, metrics in summary.items():
    em = metrics['em']
    print(f'{pipeline_name}: EM={em:.1%}')
    assert em > 0.038, f'{pipeline_name} EM {em:.1%} did not beat original 3.8% baseline'

## Appendix — Manual HF Spaces Upload

Run after Phase 6 is confirmed working. Uses the `graphcomp` secret (write-capable HF token).

Space ID: `rohanjain2312/graphbench-demo`

In [ ]:
# Cell A1 — Upload app.py to HuggingFace Space
# Run manually after benchmark is confirmed complete
from google.colab import userdata
import os

HF_WRITE_TOKEN = userdata.get('graphcomp')  # write-capable token (not the read-only HF_TOKEN)
os.environ['HF_WRITE_TOKEN'] = HF_WRITE_TOKEN

HF_SPACE_ID = 'rohanjain2312/graphbench-demo'  # NOT rohanjain2312/graphbench

from huggingface_hub import HfApi
api = HfApi(token=HF_WRITE_TOKEN)

# Upload app.py
LOCAL_APP = Path('/content/graphbench/app.py')
if LOCAL_APP.exists():
    api.upload_file(
        path_or_fileobj=str(LOCAL_APP),
        path_in_repo='app.py',
        repo_id=HF_SPACE_ID,
        repo_type='space',
    )
    print(f'✅ app.py uploaded to {HF_SPACE_ID}')
else:
    print(f'⚠️  {LOCAL_APP} not found — clone repo first')

In [ ]:
# Cell A2 — Upload GNN checkpoint to HF Space
existing_ckpts = sorted(glob.glob(str(CHECKPOINT_DIR / 'gat_*.pt')))
if existing_ckpts:
    best_ckpt = existing_ckpts[-1]
    api.upload_file(
        path_or_fileobj=best_ckpt,
        path_in_repo=f'checkpoints/{Path(best_ckpt).name}',
        repo_id=HF_SPACE_ID,
        repo_type='space',
    )
    print(f'✅ Checkpoint uploaded: {Path(best_ckpt).name}')
else:
    print('⚠️  No checkpoint found in CHECKPOINT_DIR')